In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from docx import Document

# Ensure reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Dataset directories
train_dir = 'D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\DataSet\\Train'
val_dir = 'D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\DataSet\\Val'

# Data augmentation
img_height, img_width = 224, 224
batch_size = 32

datagen_train = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, rotation_range=20, zoom_range=0.2, width_shift_range=0.2, height_shift_range=0.2
)
datagen_val = ImageDataGenerator(rescale=1./255)

train_data = datagen_train.flow_from_directory(
    train_dir, target_size=(img_height, img_width), batch_size=batch_size, class_mode='binary'
)
val_data = datagen_val.flow_from_directory(
    val_dir, target_size=(img_height, img_width), batch_size=batch_size, class_mode='binary', shuffle=False
)

# Function to save and display confusion matrix
def save_confusion_matrix(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix for {model_name}:\n{cm}")

    results_dir = 'D:\\PROJECT\\PROGRAM\\RESULTS\\'
    os.makedirs(results_dir, exist_ok=True)

    doc = Document()
    doc.add_heading(f'Confusion Matrix for {model_name}', level=1)
    for row in cm:
        doc.add_paragraph(' '.join(map(str, row)))
    doc.save(os.path.join(results_dir, f'{model_name}_confusion_matrix.docx'))

# Function to train and evaluate models
def train_and_evaluate(model, model_name):
    checkpoint_path = f'D:\\PROJECT\\PROGRAM\\MODELS\\{model_name}.keras'
    checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max')
    
    print(f"\nTraining {model_name}...\n")
    history = model.fit(train_data, epochs=20, validation_data=val_data, callbacks=[checkpoint])

    best_model = tf.keras.models.load_model(checkpoint_path)
    val_loss, val_accuracy = best_model.evaluate(val_data)
    print(f"{model_name} Validation Accuracy: {val_accuracy * 100:.2f}%")

    y_true = val_data.classes
    y_pred = (best_model.predict(val_data) > 0.5).astype('int32').flatten()

    save_confusion_matrix(model_name, y_true, y_pred)
    return val_accuracy

# CNN Model
cnn_model = Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
cnn_model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
cnn_accuracy = train_and_evaluate(cnn_model, 'CNN')

# ResNet50 Model
resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
for layer in resnet_base.layers:
    layer.trainable = True  # Fine-tune all layers

resnet_model = Model(inputs=resnet_base.input, outputs=Dense(1, activation='sigmoid')(Flatten()(resnet_base.output)))
resnet_model.compile(optimizer=Adam(learning_rate=0.00001), loss='binary_crossentropy', metrics=['accuracy'])
resnet_accuracy = train_and_evaluate(resnet_model, 'ResNet50')

# EfficientNetB0 Model
efficientnet_base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
for layer in efficientnet_base.layers:
    layer.trainable = True  # Fine-tune all layers

efficientnet_model = Model(inputs=efficientnet_base.input, outputs=Dense(1, activation='sigmoid')(Flatten()(efficientnet_base.output)))
efficientnet_model.compile(optimizer=Adam(learning_rate=0.00001), loss='binary_crossentropy', metrics=['accuracy'])
efficientnet_accuracy = train_and_evaluate(efficientnet_model, 'EfficientNetB0')

# Print Results
print("\nModel Accuracies:")
print(f"CNN Accuracy: {cnn_accuracy * 100:.2f}%")
print(f"ResNet50 Accuracy: {resnet_accuracy * 100:.2f}%")
print(f"EfficientNetB0 Accuracy: {efficientnet_accuracy * 100:.2f}%")


Found 9900 images belonging to 2 classes.
Found 825 images belonging to 2 classes.


C:\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training CNN...



C:\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 498s 2s/step - accuracy: 0.6153 - loss: 0.6522 - val_accuracy: 0.6558 - val_loss: 0.6151
Epoch 2/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 422s 1s/step - accuracy: 0.6787 - loss: 0.5947 - val_accuracy: 0.7442 - val_loss: 0.5113
Epoch 3/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 431s 1s/step - accuracy: 0.7053 - loss: 0.5566 - val_accuracy: 0.7612 - val_loss: 0.4870
Epoch 4/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 413s 1s/step - accuracy: 0.7210 - loss: 0.5389 - val_accuracy: 0.7794 - val_loss: 0.4628
Epoch 5/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 412s 1s/step - accuracy: 0.7307 - loss: 0.5245 - val_accuracy: 0.7830 - val_loss: 0.4571
Epoch 6/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 427s 1s/step - accuracy: 0.7423 - loss: 0.5094 - val_accuracy: 0.7539 - val_loss: 0.4782
Epoch 7/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 409s 1s/step - accuracy: 0.7385 - loss: 0.5134 - val_accuracy: 0.7952 - val_loss: 0.4435
Epoch 8/20
310/310 ━━━━━━━━━━━━━━━━━━━━ 452s 1s/step - accuracy: 0.7496 - loss: 0.4978 - val_accu